# 09 — Evaluación integral y equidad operativa V2

Compara las seis políticas con semillas comunes, analiza sensibilidad a capacidad y describe cobertura, intensidad, ajustes y eventos por subgrupo.

**Alcance:** simulación metodológica. Las brechas son diagnósticos operativos y no demuestran equidad ni efectividad clínica. Solo se conservan las categorías de riesgo **bajo, medio y alto**.


In [ ]:
from pathlib import Path
import subprocess, sys
REPO_URL = "https://github.com/JssSalas/pida-rl-datos-publicos.git"
BRANCH = "feature/entorno-capacidad-v2"
PROJECT_ROOT = Path("/content/pida-rl-datos-publicos")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "gymnasium", "pandas"])
if not PROJECT_ROOT.exists():
    subprocess.check_call(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT_ROOT)])
else:
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "checkout", BRANCH])
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", BRANCH])
sys.path.insert(0, str(PROJECT_ROOT))


## Configuración común

La evaluación principal y la sensibilidad reutilizan costos, horizonte y semillas. Las capacidades son escenarios operativos explícitos, no estimaciones institucionales.


In [ ]:
CAPACIDAD_MENSUAL = 100
CAPACIDADES_SENSIBILIDAD = (50, 100, 150)
COSTOS_ACCION = {0: 0, 1: 1, 2: 3, 3: 5}
HORIZONTE_MESES = 12
SEMILLAS_EVALUACION = (42, 123, 2026)


## Cohorte y modelos

Ejecute previamente los notebooks 06, 07 y 08. Este bloque no reentrena ni fabrica pacientes; carga modelos persistidos y exige que estén presentes.


In [ ]:
from google.colab import drive
import pandas as pd
drive.mount("/content/drive", force_remount=False)
ROOT = Path("/content/drive/MyDrive/PIDA-RL-Diabetes")
COHORT_PATH = ROOT / "data" / "processed" / "cohorte_diabetes.csv"
OUTPUT_DIR = ROOT / "results" / "capacidad_v2" / "evaluacion_integral"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cohorte = pd.read_csv(COHORT_PATH, dtype={"FOLIO_INT": "string"})
assert set(cohorte["categoria_riesgo"].unique()) <= {"bajo", "medio", "alto"}
MODELOS = {
    "Q-learning": ROOT / "results" / "capacidad_v2" / "q_learning" / "q_learning_v2.npz",
    "DQN": ROOT / "results" / "capacidad_v2" / "dqn" / "dqn_v2.npz",
    "PPO": ROOT / "results" / "capacidad_v2" / "ppo" / "ppo_v2.npz",
}
faltantes = [str(ruta) for ruta in MODELOS.values() if not ruta.exists()]
if faltantes:
    raise FileNotFoundError("Ejecute notebooks 06-08. Modelos faltantes: " + ", ".join(faltantes))


## Políticas comparables

Las seis políticas usan exactamente la misma cohorte, dinámica, costos, horizonte y semillas.


In [ ]:
from src.environment.diabetes_followup_env import DiabetesFollowUpEnv
from src.policies.baselines import POLITICAS_BASE
from src.policies.q_learning import QLearningAgent, crear_politica_q_learning
from src.policies.dqn import DQNAgent, crear_politica_dqn
from src.policies.ppo import PPOAgent, crear_politica_ppo
from src.evaluation.evaluate_policies import evaluar_politicas
from src.evaluation.integral import construir_reporte_integral, exportar_reporte_integral, sensibilidad_capacidad

def crear_entorno(capacidad=CAPACIDAD_MENSUAL):
    return DiabetesFollowUpEnv(cohorte, capacidad, COSTOS_ACCION, HORIZONTE_MESES)

politicas = dict(POLITICAS_BASE)
politicas["Q-learning"] = crear_politica_q_learning(QLearningAgent.cargar(MODELOS["Q-learning"], seed=20_000))
politicas["DQN"] = crear_politica_dqn(DQNAgent.cargar(MODELOS["DQN"], seed=20_000))
politicas["PPO"] = crear_politica_ppo(PPOAgent.cargar(MODELOS["PPO"], seed=20_000))
assert len(politicas) == 6


## Evaluación principal

Se conservan acciones propuestas y ejecutadas. El análisis de subgrupos se realiza después de la decisión para no cambiar la información disponible para las políticas.


In [ ]:
resultados = evaluar_politicas(lambda: crear_entorno(), politicas, SEMILLAS_EVALUACION)
sensibilidad = sensibilidad_capacidad(crear_entorno, politicas, CAPACIDADES_SENSIBILIDAD, SEMILLAS_EVALUACION)
reporte = construir_reporte_integral(resultados, cohorte, sensibilidad)
detalle = reporte["detalle"]
assert set(detalle["categoria_riesgo"]) <= {"bajo", "medio", "alto"}
assert (detalle["recursos_despues"] >= 0).all()
assert (detalle["recursos_usados_mes"] <= detalle["capacidad_mensual"]).all()
rutas = exportar_reporte_integral(reporte, OUTPUT_DIR)
display(reporte["resumen_algoritmos"].round(4))
display(reporte["estabilidad_semillas"].round(4))


## Equidad operativa

Las tablas muestran resultados por sexo codificado en la fuente, grupos de edad y nivel de riesgo. No se interpreta una diferencia como discriminación ni como efecto causal; primero debe revisarse tamaño de grupo, incertidumbre, codificación y mecanismo del simulador.


In [ ]:
display(reporte["metricas_subgrupos"].round(4))
display(reporte["brechas_equidad"].round(4))
display(reporte["sensibilidad_resumen"].round(4))
print("Archivos exportados:", len(rutas), OUTPUT_DIR)


## Criterios de lectura responsable

- No seleccione una política únicamente por recompensa.
- Revise cobertura de riesgo alto, eventos, recursos, ajustes, estabilidad y brechas conjuntamente.
- Compare acción propuesta contra ejecutada para identificar demanda no atendida.
- Si cambian cohorte, recompensa, costos o capacidad, reentrene Q-learning, DQN y PPO.
- La validación con simulación no sustituye validación clínica, ética ni institucional.
